# Counterfactuals — ADNI

DDIM inversion + per-attribute intervention on ADNI MRI slices using the
six-attribute SCM (apoE, age, sex, brain_vol, vent_vol, slice). Common
pipeline setup is shared via `inference_utils.py`.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('/projects/dsai/se_aieng/cai/causal/workspace/Causal-Adapter/causal-adapter-sd15')

import os
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torchvision import transforms

from causal_modules.ddim_modules import sample, ddim_editing, save_images_grid
from causal_modules.control_heads import bin_array, ordinal_array
from causal_datasets.adni_dataset import load_data, load_extra_attributes
from causal_datasets._normalization import normalize as adni_normalize
from inference_utils import build_transforms, load_causal_adapter

## 1. Configuration

In [ ]:
DATASET = 'ADNI'
SIZE = 256

BASE_MODEL_PATH = '/projects/dsai/se_aieng/cai/causal/workspace/causaledit/MCPL-diffuser/.cache/huggingface/hub/models--lambdalabs--miniSD-diffusers/snapshots/26ed8a9bfbf76f46a6cf60517dde321f900c44ce'

RUN_DIR = '/projects/dsai/se_aieng/cai/causal/workspace/causaledit/MCPL-diffuser/logs/logs_ADNI_all/2025-05-06T14-33-14-controlnet_textcond_contrastgeneration_text_global_after'
STEPS = 100000
CONTROLNET_PATH     = f'{RUN_DIR}/controlnet-steps-{STEPS}.safetensors'
TEXT_EMBEDDING_PATH = f'{RUN_DIR}/learned_embeds-steps-{STEPS}.safetensors'
SCM_PATH            = '/projects/dsai/se_aieng/cai/causal/workspace/causaledit/MCPL-diffuser/logs/logs_ADNI_all/2025-05-01T09-58-53-causalnet_pretrain/best_model.pt'

DATA_ROOT = '/projects/dsai/se_aieng/cai/causal/workspace/causaledit/counterfactual-benchmark/counterfactual_benchmark/ctf_datasets/adni/preprocessing'

## 2. Build pipeline + transforms

In [ ]:
image_transforms, original_transforms, _ = build_transforms(DATASET, size=SIZE)

assets = load_causal_adapter(
    DATASET,
    base_model_path=BASE_MODEL_PATH,
    controlnet_path=CONTROLNET_PATH,
    text_embedding_path=TEXT_EMBEDDING_PATH,
    scm_path=SCM_PATH,
)
pipe              = assets.pipe
device            = assets.device
prompt            = assets.prompt  # already padded to 10 trailing chars by helper
presudo_words     = assets.presudo_words
presudo_list      = assets.presudo_list
presudo_token_ids = assets.presudo_token_ids

## 3. Load an ADNI slice + attributes

In [ ]:
ATTRIBUTE_SIZE = {
    'apoE': 2, 'age': 1, 'sex': 1, 'brain_vol': 1, 'vent_vol': 1, 'slice': 10,
}


def load_adni_split(data_root=DATA_ROOT, split='test', num_of_slices=10):
    data_dir = os.path.join(data_root, 'preprocessed_data')
    image_paths, attribute_dict, subject_dates_dict = load_data(
        data_dir, num_of_slices=num_of_slices, split=split,
        keep_only_screening=False,
    )
    csv_path = list(Path(data_root).glob('ADNIMERGE*.csv'))[0]
    assert csv_path.is_file(), 'Provide ADNIMERGE csv path'

    attributes, indices_to_remove = load_extra_attributes(
        csv_path, attributes=ATTRIBUTE_SIZE.keys(),
        attribute_dict=attribute_dict, subject_dates_dict=subject_dates_dict,
        keep_only_screening=False,
    )
    attributes['slice'] = np.delete(attributes['slice'], indices_to_remove, axis=0)
    attributes = {
        attr: adni_normalize(torch.tensor(np.array(values), dtype=torch.float32), attr)
        for attr, values in attributes.items()
    }
    attrs = torch.cat([
        attributes[attr].unsqueeze(1) if attributes[attr].ndim == 1 else attributes[attr]
        for attr in ATTRIBUTE_SIZE.keys()
    ], dim=1)
    image_paths = [p for i, p in enumerate(image_paths) if i not in indices_to_remove]
    return image_paths, attrs


img_paths, imglabel = load_adni_split()
print(f'loaded {len(img_paths)} ADNI slices')

In [ ]:
img_id = 10  # set to ``np.random.randint(0, len(img_paths))`` for a fresh sample
arr_uint8 = (img_paths[img_id] * 255).clip(0, 255).astype(np.uint8)
img = Image.fromarray(arr_uint8, mode='L')
label = imglabel[img_id].unsqueeze(0)
print('label:', label)
img

## 4. DDIM inversion + reconstruction

In [ ]:
image = img.convert('RGB') if img.mode != 'RGB' else img
original_img = original_transforms(image.copy())
condition_image = image.copy()
image_t = image_transforms(image)

set_guidance_scale = 1.0
num_steps = 50

final_im, inverted_latents, _, _ = ddim_editing(
    pipe, image_t.unsqueeze(0), label.clone(), presudo_token_ids, prompt,
    num_steps=num_steps,
    invert_guidance_scale=1.0,
    set_guidance_scale=set_guidance_scale,
    intervention_indx=None,
    intervention_values=None,
    return_PIL=True,
)
save_images_grid([[original_img, final_im[0]]], (1, 2), None)

## 5. Direct-effect helpers

Visualises pixel-wise residuals between the source and intervened images
using a diverging colormap (RdBu_r), with a small per-cell colorbar.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def plot_results(interved_images, direct_effects, titles=None, threshold=5.0):
    n_cols = len(interved_images)
    if titles is None:
        titles = [f'Image {i}' for i in range(n_cols)]

    fig, axes = plt.subplots(
        2, n_cols, figsize=(1.9 * n_cols, 4.1),
        gridspec_kw={'wspace': -0.08, 'hspace': 0.015},
    )
    if n_cols == 1:
        axes = axes.reshape(2, 1)

    for i, im in enumerate(interved_images):
        ax = axes[0, i]
        ax.imshow(np.array(im), cmap='gray')
        ax.axis('off')
        ax.set_anchor('S')

    ims = []
    for i, im in enumerate(direct_effects):
        ax = axes[1, i]
        ax.axis('off')
        ax.set_anchor('N')
        if im is None:
            continue
        effect = im.copy()
        effect[np.abs(effect) < threshold] = 0.0
        lim = np.abs(effect).max() or 1e-6
        h = ax.imshow(effect, cmap='RdBu_r', vmin=-lim, vmax=lim)
        ims.append((i, h))
        if i > 0:
            rect = patches.Rectangle(
                (0, 0), 1, 1, transform=ax.transAxes,
                linewidth=1.5, edgecolor='black', facecolor='none',
            )
            ax.add_patch(rect)

    fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.10,
                        wspace=0.02, hspace=0.02)
    fig.canvas.draw()

    for i, h in ims:
        ax = axes[1, i]
        pos = ax.get_position()
        cax = fig.add_axes([pos.x0, pos.y0 - 0.030, pos.width, 0.018])
        cbar = fig.colorbar(h, cax=cax, orientation='horizontal')
        cbar.ax.tick_params(labelsize=7, length=2, pad=1)
    plt.show()


def direct_effect(inter_img, ori_img):
    if not isinstance(inter_img, Image.Image):
        inter_img = Image.fromarray(inter_img)
    if not isinstance(ori_img, Image.Image):
        ori_img = Image.fromarray(ori_img)
    return (
        np.array(inter_img.convert('L'), dtype=np.float32)
        - np.array(ori_img.convert('L'), dtype=np.float32)
    )

## 6. Per-attribute intervention sweep

Each attribute has a different valid intervention space:

* `apoE` (idx 0): categorical {0,1,2} -> binary array.
* `sex` (idx 2): binary -> complement of source label.
* `slice` (idx 5): ordinal {1,...,10} -> one-hot via ``ordinal_array``.
* others (age/brain_vol/vent_vol): continuous in [0, 1].

In [ ]:
fixed_seeds = False
if fixed_seeds:
    seed = 10
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
else:
    torch.seed()

image_lists = [original_img, final_im[0]]
residual_maps = [None, direct_effect(final_im[0], original_img)]
causal_cond_list = [label]

for inter_id in range(6):
    if inter_id == 0:
        gt_attr = label[:, 0]
        rand_vals = torch.randint(0, 3, (label.shape[0],), device=label.device)
        mask = rand_vals == gt_attr
        while mask.any():
            rand_vals[mask] = torch.randint(0, 3, (int(mask.sum().item()),), device=label.device)
            mask = rand_vals == gt_attr
        inter_value = bin_array(rand_vals, m=2, reverse=False)
    elif inter_id == 2:
        inter_value = 1 - label[:, 3]  # mirrors the original notebook's choice
    elif inter_id == 5:
        rand_vals = torch.randint(1, 11, (label.shape[0],), device=label.device)
        inter_value = ordinal_array(rand_vals, m=10, reverse=False)
    else:
        inter_value = torch.rand(label.shape[0], device=label.device)

    interved_image, causal_cond = sample(
        pipe, prompt, presudo_token_ids,
        start_step=0,
        start_latents=inverted_latents[-1].clone(),
        guidance_scale=1.0,
        num_inference_steps=50,
        num_images_per_prompt=1,
        negative_prompt=None,
        device=device,
        controlnet_image=None,
        intervention_indx=inter_id,
        intervention_values=inter_value,
        label=label.clone(),
        return_PIL=True,
    )
    causal_cond_list.append(causal_cond.squeeze(2))
    edited = interved_image[0]
    image_lists.append(edited)
    residual_maps.append(direct_effect(edited, original_img))

plot_results(image_lists, residual_maps)

## 7. Continuous traversal on `age`

In [ ]:
range_len = 30
values = torch.from_numpy(np.linspace(0, 1.0, num=range_len))

image_lists = [original_img]
for v in values:
    interved_image, _ = sample(
        pipe, prompt, presudo_token_ids,
        start_step=0,
        start_latents=inverted_latents[-1].clone(),
        guidance_scale=1.0,
        num_inference_steps=num_steps,
        num_images_per_prompt=1,
        negative_prompt=None,
        device=device,
        controlnet_image=None,
        intervention_indx=1,
        intervention_values=v,
        label=label.clone(),
        return_PIL=True,
    )
    image_lists.append(interved_image[0])

save_images_grid([image_lists], (1, len(image_lists)), './intervention_ADNI_age.png')
print('saved ./intervention_ADNI_age.png')

## 8. Cross-attention maps

In [ ]:
from causal_modules.p2p_edits import ptp_tools, ptp_utils
import importlib
importlib.reload(ptp_tools)
importlib.reload(ptp_utils)

out_dir = os.path.join('./textcond', prompt)
os.makedirs(out_dir, exist_ok=True)

generator = torch.manual_seed(0)
overlapped_mask, attn_img = ptp_tools.plot_img_attn_mask_textcontrol(
    pipe, [prompt], presudo_words, presudo_token_ids, condition_image,
    device, out_dir, 'causalnet.png',
    latent=image_t.unsqueeze(0), res=16, label=label,
    GUIDANCE_SCALE=2.0, attn_threshold=0.5, only_sampling=False,
    dataset='ADNI', show_text=True, save_masks=False, class_select=False,
    intervention_indx=None, intervention_values=None,
    from_where=['down', 'up'], mask_concepts=True,
    g_gpu=generator, num_steps=30, img_size=SIZE, exp_names=['textcond'],
)